# RUKOPYS Qwen3-VL OCR Stage 1: Silver Warm-up

OCR-only LoRA training for the YOLO pipeline. It trains on cropped text regions from the silver split and adds jittered crop boxes to mimic detector noise.

In [ ]:
INSTALL_DEPS = True

if INSTALL_DEPS:
    import subprocess
    import sys

    commands = [
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "--upgrade-strategy",
            "only-if-needed",
            "accelerate",
            "datasets",
            "peft",
            "bitsandbytes",
            "qwen-vl-utils",
            "trl",
            "pandas==2.2.2",
            "pillow<12",
        ],
        [sys.executable, "-m", "pip", "install", "-q", "-U", "git+https://github.com/huggingface/transformers.git"],
    ]
    for cmd in commands:
        print("Running:", " ".join(cmd), flush=True)
        subprocess.check_call(cmd)


In [ ]:
import gc
import json
import math
import os
import random
import re
import time
from collections import Counter, defaultdict
from pathlib import Path

import torch
from datasets import Dataset
from PIL import Image

Image.MAX_IMAGE_PIXELS = None
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"

BASE_MODEL_CANDIDATES = [
    "/kaggle/input/models/qwen-lm/qwen-3-vl/transformers/8b-instruct/1",
    "/kaggle/input/qwen3-vl-8b-instruct",
    "Qwen/Qwen3-VL-8B-Instruct",
]

DATASET_ROOT = "/kaggle/input/datasets/quii29/rukopys-dataset"
PREFER_PRE_CROPPED_DATASET = True
PRE_CROPPED_STAGE1_SPLIT_CANDIDATES = ["silver", "train"]
CROPPED_DATASET_ROOT_CANDIDATES = [
    "/kaggle/input/cropped-rukopys-dataset",
    "/kaggle/input/cropped-rukopys-dataset/Cropped Rukopys Dataset",
    "/kaggle/input/cropped-rukopys",
    "/kaggle/input/rukopys-cropped",
    "/kaggle/input/rukopys-cropped-dataset",
]
OUTPUT_DIR = Path("/kaggle/working/qwen3vl_yolo_ocr_stage1_silver")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42
VAL_RATIO = 0.03
MAX_TEXT_CHARS = 500
MAX_PIXELS_CROP = 200_000
CROP_PAD_RATIO = 0.04
JITTER_VARIANTS_PER_REGION = 1
JITTER_SHIFT_RATIO = 0.08
JITTER_SCALE_RATIO = 0.12
MAX_TRAIN_SAMPLES = None  # optional global cap after type-balanced sampling
MAX_VAL_SAMPLES = 100

# Stage 1 is only a silver warm-up, so keep it small and type-balanced.
# With pre-cropped data these are crop-row limits per type; with original metadata they are bbox-region limits.
# Fast Kaggle preset: about 1.2k crops => roughly 150 optimizer steps with batch=1 and grad_accum=8.
STAGE1_TYPE_REGION_LIMITS = {
    "handwritten": 600,
    "formula": 250,
    "table": 150,
    "annotation": 150,
    "printed": 100,
}

PER_DEVICE_BATCH = 1
GRAD_ACCUM = 8
LEARNING_RATE = 2e-5
NUM_TRAIN_EPOCHS = 1.0
SAVE_TOTAL_LIMIT = 2
LOGGING_STEPS = 1
EVAL_STEPS = 500
SAVE_STEPS = 100

VALID_TYPES = {"handwritten", "printed", "formula", "table", "annotation", "image", "graph"}
TYPE_ALIASES = {
    "handwriting": "handwritten",
    "handwritten_text": "handwritten",
    "text": "handwritten",
    "printed_text": "printed",
    "print": "printed",
    "math": "formula",
    "equation": "formula",
    "chemical": "formula",
    "tabular": "table",
    "note": "annotation",
    "mark": "annotation",
}
TEXT_TYPES = {"handwritten", "printed", "formula", "table", "annotation"}
IMAGE_EXTENSIONS = [".png", ".jpg", ".jpeg", ".webp", ".bmp"]

COMMON_OCR_RULES = (
    "Return only the transcription. No JSON, no Markdown, no explanation. "
    "Preserve the original script, spelling, punctuation, capitalization, digits, "
    "abbreviations, quotes, hyphens, and line-final dashes. "
    "Do not translate, correct grammar, normalize spelling, expand abbreviations, "
    "or infer hidden text. Use [illegible] only for truly unreadable words."
)

CROP_PROMPTS = {
    "handwritten": (
        "Transcribe this cropped handwritten text line from a Ukrainian document exactly as visible. "
        + COMMON_OCR_RULES
    ),
    "printed": (
        "Transcribe this cropped printed or typed text line exactly as visible. "
        + COMMON_OCR_RULES
    ),
    "annotation": (
        "Transcribe this short teacher annotation, grade, mark, correction, or numbering exactly as visible. "
        + COMMON_OCR_RULES
    ),
    "formula": (
        "Transcribe only the standalone math or chemistry expression. "
        "Prefer concise LaTeX for fractions, roots, superscripts, subscripts, matrices, arrows, and chemical notation. "
        "Do not wrap the answer in dollar signs. No explanation."
    ),
    "table": (
        "Transcribe this cropped table. Return rows in reading order, one row per line, "
        "with cells separated by |. Do not create a Markdown table. No explanation."
    ),
    "default": "Transcribe the visible content exactly. " + COMMON_OCR_RULES,
}


In [ ]:
def first_existing(paths):
    for item in paths:
        p = Path(item)
        if p.exists():
            return p
    return None


def find_model_id():
    for item in BASE_MODEL_CANDIDATES:
        if item.startswith("/") and Path(item).exists():
            return item
        if not item.startswith("/"):
            return item
    raise FileNotFoundError("No Qwen3-VL base model found.")


def normalize_type(value):
    value = str(value or "handwritten").strip().lower().replace("-", "_").replace(" ", "_")
    value = TYPE_ALIASES.get(value, value)
    return value if value in VALID_TYPES else "handwritten"


def read_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def metadata_looks_precropped(path):
    try:
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                row = json.loads(line)
                return "image" in row and "text" in row and ("label" in row or "type" in row)
    except Exception:
        return False
    return False


def iter_precropped_roots():
    seen = set()

    def emit(path):
        path = Path(path)
        try:
            key = str(path.resolve())
        except Exception:
            key = str(path)
        if key not in seen and path.exists() and path.is_dir():
            seen.add(key)
            return path
        return None

    for item in CROPPED_DATASET_ROOT_CANDIDATES:
        root = emit(item)
        if root is not None:
            yield root

    input_root = Path("/kaggle/input")
    if not input_root.exists():
        return
    for parent in sorted([p for p in input_root.iterdir() if p.is_dir()]):
        root = emit(parent)
        if root is not None:
            yield root
        try:
            children = sorted([p for p in parent.iterdir() if p.is_dir()])
        except Exception:
            children = []
        for child in children:
            root = emit(child)
            if root is not None:
                yield root


def find_precropped_metadata(split_candidates):
    for root in iter_precropped_roots():
        for split in split_candidates:
            meta = root / split / "metadata.jsonl"
            if meta.exists() and metadata_looks_precropped(meta):
                return root, split, meta
    return None, None, None


def resolve_image_path(root, split, file_name):
    root = Path(root)
    raw = Path(file_name)
    name = raw.name
    stem = raw.stem
    candidate_names = [name] + [stem + ext for ext in IMAGE_EXTENSIONS if stem + ext != name]
    candidates = [root / split / file_name, root / file_name]
    for candidate_name in candidate_names:
        candidates.extend([root / split / "images" / candidate_name, root / split / candidate_name])
    for p in candidates:
        if p.exists():
            return str(p)
    return str(root / split / "images" / candidate_names[0])


def resolve_crop_path(root, split, image_name):
    root = Path(root)
    raw = Path(str(image_name or ""))
    if raw.is_absolute() and raw.exists():
        return str(raw)
    candidates = [
        root / split / raw,
        root / raw,
        root / split / "images" / raw.name,
        root / "images" / raw.name,
    ]
    for p in candidates:
        if p.exists():
            return str(p)
    return str(candidates[0])


def resize_to_pixel_budget(image, max_pixels=MAX_PIXELS_CROP):
    total = max(1, image.size[0] * image.size[1])
    if total <= max_pixels:
        return image
    scale = (max_pixels / total) ** 0.5
    new_size = (max(1, int(image.size[0] * scale)), max(1, int(image.size[1] * scale)))
    return image.resize(new_size, Image.Resampling.LANCZOS)


def load_sample_image(sample):
    crop_path = sample.get("crop_path")
    if crop_path:
        with Image.open(crop_path) as img:
            img = img.convert("RGB").copy()
        return resize_to_pixel_budget(img)
    return crop_with_padding(sample["image_path"], sample["bbox"])


def clamp_box(box, width, height):
    if not isinstance(box, (list, tuple)) or len(box) != 4:
        return None
    try:
        x1, y1, x2, y2 = [float(v) for v in box]
    except Exception:
        return None
    x1, x2 = sorted((max(0, min(width, x1)), max(0, min(width, x2))))
    y1, y2 = sorted((max(0, min(height, y1)), max(0, min(height, y2))))
    if x2 - x1 < 3 or y2 - y1 < 3:
        return None
    return [int(round(x1)), int(round(y1)), int(round(x2)), int(round(y2))]


def jitter_box(box, width, height, rng):
    x1, y1, x2, y2 = [float(v) for v in box]
    bw = max(1.0, x2 - x1)
    bh = max(1.0, y2 - y1)
    shift_x = rng.uniform(-JITTER_SHIFT_RATIO, JITTER_SHIFT_RATIO) * bw
    shift_y = rng.uniform(-JITTER_SHIFT_RATIO, JITTER_SHIFT_RATIO) * bh
    scale_x = 1.0 + rng.uniform(-JITTER_SCALE_RATIO, JITTER_SCALE_RATIO)
    scale_y = 1.0 + rng.uniform(-JITTER_SCALE_RATIO, JITTER_SCALE_RATIO)
    cx = (x1 + x2) / 2 + shift_x
    cy = (y1 + y2) / 2 + shift_y
    new_w = bw * scale_x
    new_h = bh * scale_y
    return clamp_box([cx - new_w / 2, cy - new_h / 2, cx + new_w / 2, cy + new_h / 2], width, height)


def crop_with_padding(image_path, bbox, pad_ratio=CROP_PAD_RATIO):
    with Image.open(image_path) as img:
        img = img.convert("RGB")
        width, height = img.size
        x1, y1, x2, y2 = bbox
        pad = int(round(max(x2 - x1, y2 - y1) * pad_ratio))
        x1 = max(0, x1 - pad)
        y1 = max(0, y1 - pad)
        x2 = min(width, x2 + pad)
        y2 = min(height, y2 + pad)
        crop = img.crop((x1, y1, x2, y2))
        return resize_to_pixel_budget(crop)


def make_precropped_ocr_samples(records, root, split, seed):
    samples = []
    skipped = Counter()
    for row_idx, record in enumerate(records):
        text = str(record.get("text") or "").strip()
        if not text:
            skipped["empty_text"] += 1
            continue
        if len(text) > MAX_TEXT_CHARS:
            skipped["long_text"] += 1
            continue
        rtype = normalize_type(record.get("label", record.get("type", "handwritten")))
        if rtype not in TEXT_TYPES:
            skipped["non_text"] += 1
            continue
        image_name = record.get("image") or record.get("file_name") or record.get("path")
        crop_path = resolve_crop_path(root, split, image_name)
        if not Path(crop_path).exists():
            skipped["missing_crop"] += 1
            continue
        prompt = CROP_PROMPTS.get(rtype, CROP_PROMPTS["default"])
        samples.append({
            "crop_path": crop_path,
            "image_path": crop_path,
            "bbox": None,
            "region_type": rtype,
            "prompt": prompt,
            "answer": text,
            "source": f"pre_cropped_{split}",
            "file_name": str(image_name or ""),
            "region_index": row_idx,
            "crop_variant": "pre_cropped",
        })
    return samples, skipped


def make_ocr_samples(records, root, split, seed):
    rng = random.Random(seed)
    samples = []
    skipped = Counter()
    for record in records:
        image_path = resolve_image_path(root, split, record["file_name"])
        width = int(record.get("image_width") or 0)
        height = int(record.get("image_height") or 0)
        if width <= 0 or height <= 0:
            try:
                with Image.open(image_path) as img:
                    width, height = img.size
            except Exception:
                skipped["bad_image"] += 1
                continue
        for region_idx, region in enumerate(record.get("regions") or []):
            rtype = normalize_type(region.get("type"))
            if rtype not in TEXT_TYPES:
                skipped["non_text"] += 1
                continue
            text = str(region.get("text") or "").strip()
            if not text:
                skipped["empty_text"] += 1
                continue
            if len(text) > MAX_TEXT_CHARS:
                skipped["long_text"] += 1
                continue
            box = clamp_box(region.get("bbox"), width, height)
            if box is None:
                skipped["bad_box"] += 1
                continue
            prompt = CROP_PROMPTS.get(rtype, CROP_PROMPTS["default"])
            boxes = [("gt", box)]
            for j in range(JITTER_VARIANTS_PER_REGION):
                jittered = jitter_box(box, width, height, rng)
                if jittered is not None:
                    boxes.append((f"jitter{j + 1}", jittered))
            for variant, crop_box in boxes:
                samples.append({
                    "image_path": image_path,
                    "bbox": crop_box,
                    "region_type": rtype,
                    "prompt": prompt,
                    "answer": text,
                    "source": record.get("source", "unknown"),
                    "file_name": record.get("file_name", ""),
                    "region_index": region_idx,
                    "crop_variant": variant,
                })
    return samples, skipped


def count_regions_by_type(samples):
    groups = {}
    for item in samples:
        key = (item.get("region_type"), item.get("file_name"), item.get("region_index"))
        groups[key] = item.get("region_type", "unknown")
    return Counter(groups.values())


def cap_samples_by_type_region(samples, limits, seed=RANDOM_SEED):
    if not limits:
        return samples, {}
    rng = random.Random(seed)
    grouped = defaultdict(list)
    for item in samples:
        key = (item.get("file_name"), item.get("region_index"))
        grouped[(item.get("region_type", "unknown"), key)].append(item)

    by_type = defaultdict(list)
    for (rtype, key), items in grouped.items():
        by_type[rtype].append(items)

    selected = []
    report = {}
    for rtype, groups in sorted(by_type.items()):
        rng.shuffle(groups)
        limit = limits.get(rtype, len(groups))
        kept_groups = groups[: min(limit, len(groups))]
        selected.extend(item for group in kept_groups for item in group)
        report[rtype] = {
            "available_regions": len(groups),
            "kept_regions": len(kept_groups),
            "kept_samples": sum(len(group) for group in kept_groups),
            "limit": limit,
        }
    rng.shuffle(selected)
    return selected, report


def split_samples(samples, val_ratio=VAL_RATIO, seed=RANDOM_SEED):
    rng = random.Random(seed)
    by_source = defaultdict(list)
    for item in samples:
        by_source[item.get("source", "unknown")].append(item)
    train, val = [], []
    for source, items in by_source.items():
        rng.shuffle(items)
        n_val = max(1, int(len(items) * val_ratio)) if len(items) >= 20 else 0
        val.extend(items[:n_val])
        train.extend(items[n_val:])
        print(f"source={source}: train={len(items[n_val:])} val={len(items[:n_val])}")
    rng.shuffle(train)
    rng.shuffle(val)
    if MAX_TRAIN_SAMPLES is not None:
        train = train[:MAX_TRAIN_SAMPLES]
    if MAX_VAL_SAMPLES is not None:
        val = val[:MAX_VAL_SAMPLES]
    return train, val


In [ ]:
dataset_mode = "bbox_crop"
metadata_path = None
root = Path(DATASET_ROOT)

if PREFER_PRE_CROPPED_DATASET:
    cropped_root, cropped_split, cropped_meta = find_precropped_metadata(PRE_CROPPED_STAGE1_SPLIT_CANDIDATES)
else:
    cropped_root, cropped_split, cropped_meta = None, None, None

if cropped_meta is not None:
    dataset_mode = "pre_cropped"
    root = Path(cropped_root)
    silver_split = cropped_split
    metadata_path = str(cropped_meta)
    silver_records = read_jsonl(cropped_meta)
    print(f"Using pre-cropped OCR dataset: root={root} split={silver_split} records={len(silver_records)}")
    samples, skipped = make_precropped_ocr_samples(silver_records, root, silver_split, RANDOM_SEED)
else:
    silver_meta = root / "silver" / "metadata.jsonl"
    if not silver_meta.exists():
        silver_meta = root / "sliver" / "metadata.jsonl"
    if not silver_meta.exists():
        raise FileNotFoundError(f"No pre-cropped dataset found and no silver metadata found under {root}")
    silver_split = silver_meta.parent.name
    metadata_path = str(silver_meta)
    silver_records = read_jsonl(silver_meta)
    print(f"Using original bbox metadata: root={root} split={silver_split} records={len(silver_records)}")
    samples, skipped = make_ocr_samples(silver_records, root, silver_split, RANDOM_SEED)

print("Dataset mode:", dataset_mode)
print("OCR samples before cap:", len(samples))
print("Rows/regions by type before cap:", count_regions_by_type(samples))
print("Skipped:", skipped)

samples, type_cap_report = cap_samples_by_type_region(samples, STAGE1_TYPE_REGION_LIMITS, RANDOM_SEED)
print("Type cap report:", json.dumps(type_cap_report, ensure_ascii=False, indent=2))
print("OCR samples after cap:", len(samples))
print("Rows/regions by type after cap:", count_regions_by_type(samples))

train_rows, val_rows = split_samples(samples)
print("Train samples:", len(train_rows), "Val samples:", len(val_rows))
print("Estimated optimizer steps:", math.ceil(len(train_rows) / max(1, PER_DEVICE_BATCH * GRAD_ACCUM)))

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "ocr_stage1_train_sample_count.json").write_text(
    json.dumps({
        "dataset_mode": dataset_mode,
        "dataset_root": str(root),
        "metadata_path": metadata_path,
        "train": len(train_rows),
        "val": len(val_rows),
        "skipped": dict(skipped),
        "type_cap_report": type_cap_report,
        "type_region_limits": STAGE1_TYPE_REGION_LIMITS,
        "estimated_optimizer_steps": math.ceil(len(train_rows) / max(1, PER_DEVICE_BATCH * GRAD_ACCUM)),
    }, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
train_ds = Dataset.from_list(train_rows)
val_ds = Dataset.from_list(val_rows)
print(train_ds[0])


In [ ]:
from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training
from qwen_vl_utils import process_vision_info
from transformers import AutoModelForImageTextToText, AutoProcessor, BitsAndBytesConfig, TrainerCallback
from trl import SFTConfig, SFTTrainer


def configure_processor(processor):
    if processor.tokenizer.pad_token_id is None:
        processor.tokenizer.pad_token = processor.tokenizer.eos_token
    processor.tokenizer.padding_side = "left"
    return processor


def force_fp16_config(model):
    model.config.torch_dtype = torch.float16
    for attr in ("text_config", "vision_config"):
        cfg = getattr(model.config, attr, None)
        if cfg is not None:
            cfg.torch_dtype = torch.float16
            if hasattr(cfg, "dtype"):
                cfg.dtype = "float16"


def make_lora_config():
    kwargs = dict(
        r=64,
        lora_alpha=64,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
    )
    try:
        return LoraConfig(**kwargs, use_rslora=True)
    except TypeError:
        return LoraConfig(**kwargs)


class OCRDataCollator:
    def __init__(self, processor):
        self.processor = processor
        self.assistant_prefix = processor.tokenizer.encode(
            "<|im_start|>assistant\n", allowed_special="all", add_special_tokens=False
        )

    def build_messages(self, sample):
        crop = load_sample_image(sample)
        return [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": crop},
                    {"type": "text", "text": sample["prompt"]},
                ],
            },
            {"role": "assistant", "content": [{"type": "text", "text": sample["answer"]}]},
        ]

    def __call__(self, examples):
        messages = [self.build_messages(ex) for ex in examples]
        texts = [self.processor.apply_chat_template(m, tokenize=False, add_generation_prompt=False) for m in messages]
        image_inputs, video_inputs = process_vision_info(messages)
        try:
            batch = self.processor(
                text=texts,
                images=image_inputs,
                videos=video_inputs,
                text_kwargs={"padding": True, "return_tensors": "pt"},
                images_kwargs={"return_tensors": "pt"},
                videos_kwargs={"return_tensors": "pt"},
            )
        except TypeError:
            batch = self.processor(text=texts, images=image_inputs, videos=video_inputs, padding=True, return_tensors="pt")
        labels = batch["input_ids"].clone()
        labels[labels == self.processor.tokenizer.pad_token_id] = -100
        prefix = torch.tensor(self.assistant_prefix, device=labels.device)
        prefix_len = len(self.assistant_prefix)
        for i in range(labels.size(0)):
            if labels[i].size(0) < prefix_len:
                labels[i, :] = -100
                continue
            windows = labels[i].unfold(0, prefix_len, 1)
            matches = (windows == prefix).all(dim=1)
            idxs = matches.nonzero(as_tuple=True)[0]
            if len(idxs):
                labels[i, : idxs[0].item() + prefix_len] = -100
            else:
                labels[i, :] = -100
        batch["labels"] = labels
        return batch


class PrintProgressCallback(TrainerCallback):
    def __init__(self, name, bar_width=20, heartbeat_substeps=20):
        self.name = name
        self.bar_width = bar_width
        self.heartbeat_substeps = heartbeat_substeps
        self.start_time = None
        self.substeps = 0
        self.last_step_printed = -1

    @staticmethod
    def format_duration(seconds):
        if seconds is None or seconds <= 0:
            return "--:--"
        seconds = int(round(seconds))
        hours = seconds // 3600
        minutes = (seconds % 3600) // 60
        secs = seconds % 60
        if hours:
            return f"{hours:02d}:{minutes:02d}:{secs:02d}"
        return f"{minutes:02d}:{secs:02d}"

    def make_bar(self, step, total):
        ratio = step / max(1, total)
        filled = min(self.bar_width, max(0, int(round(self.bar_width * ratio))))
        return "█" * filled + " " * (self.bar_width - filled)

    def progress_text(self, state, loss="--", lr="--", suffix=""):
        total = max(1, state.max_steps)
        step = min(total, max(0, state.global_step))
        elapsed = time.time() - (self.start_time or time.time())
        speed = step / max(1e-6, elapsed) if step > 0 else 0.0
        eta = (total - step) / max(1e-6, speed) if speed > 0 else None
        pct = 100.0 * step / total
        epoch = state.epoch if state.epoch is not None else "--"
        if isinstance(loss, float):
            loss = f"{loss:.4f}"
        if isinstance(lr, float):
            lr = f"{lr:.2e}"
        if isinstance(epoch, float):
            epoch = f"{epoch:.2f}"
        if speed > 0:
            timing = f"{self.format_duration(elapsed)}<{self.format_duration(eta)}, {speed:.3f} step/s"
        else:
            timing = f"{self.format_duration(elapsed)}<?, -- step/s"
        return (
            f"{self.name}: {pct:3.0f}%|{self.make_bar(step, total)}| "
            f"{step}/{total} [{timing}, loss={loss}, lr={lr}, epoch={epoch}{suffix}]"
        )

    def on_train_begin(self, args, state, control, **kwargs):
        self.start_time = time.time()
        self.substeps = 0
        print(self.progress_text(state, suffix=", start"), flush=True)

    def on_substep_end(self, args, state, control, **kwargs):
        self.substeps += 1
        if self.heartbeat_substeps and self.substeps % self.heartbeat_substeps == 0:
            print(
                self.progress_text(
                    state,
                    suffix=f", micro={self.substeps}, grad_accum={args.gradient_accumulation_steps}",
                ),
                flush=True,
            )

    def on_step_end(self, args, state, control, **kwargs):
        if state.global_step != self.last_step_printed:
            self.last_step_printed = state.global_step
            print(self.progress_text(state, suffix=", step_end"), flush=True)

    def on_log(self, args, state, control, logs=None, **kwargs):
        logs = logs or {}
        loss = logs.get("loss", logs.get("eval_loss", "--"))
        lr = logs.get("learning_rate", "--")
        print(self.progress_text(state, loss=loss, lr=lr), flush=True)


class OOMRecoverySFTTrainer(SFTTrainer):
    def training_step(self, model, inputs, num_items_in_batch=None):
        try:
            try:
                return super().training_step(model, inputs, num_items_in_batch=num_items_in_batch)
            except TypeError:
                return super().training_step(model, inputs)
        except torch.cuda.OutOfMemoryError:
            print("OOM during training step; clearing cache and skipping this batch.", flush=True)
            torch.cuda.empty_cache()
            gc.collect()
            return torch.tensor(0.0, device=model.device, requires_grad=True)


model_id = find_model_id()
print("Base model:", model_id)
processor = configure_processor(AutoProcessor.from_pretrained(model_id, trust_remote_code=True))

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)
base_model = AutoModelForImageTextToText.from_pretrained(
    model_id,
    device_map="auto",
    quantization_config=quantization_config,
    dtype=torch.float16,
    trust_remote_code=True,
    attn_implementation="sdpa",
    low_cpu_mem_usage=True,
)
force_fp16_config(base_model)
base_model = prepare_model_for_kbit_training(base_model, use_gradient_checkpointing=True)
model = get_peft_model(base_model, make_lora_config())
model.print_trainable_parameters()

data_collator = OCRDataCollator(processor)

training_args = SFTConfig(
    output_dir=str(OUTPUT_DIR / "stage1_silver_ocr"),
    per_device_train_batch_size=PER_DEVICE_BATCH,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    num_train_epochs=NUM_TRAIN_EPOCHS,
    optim="paged_adamw_8bit",
    max_grad_norm=0.3,
    warmup_ratio=0.03,
    logging_steps=LOGGING_STEPS,
    eval_strategy="steps",
    eval_steps=EVAL_STEPS,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=SAVE_TOTAL_LIMIT,
    report_to="none",
    remove_unused_columns=False,
    gradient_checkpointing=True,
    dataset_kwargs={"skip_prepare_dataset": True},
    dataloader_num_workers=0,
)

trainer = OOMRecoverySFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=data_collator,
    callbacks=[PrintProgressCallback("ocr-stage1-silver")],
)
trainer.train(resume_from_checkpoint=False)

final_dir = OUTPUT_DIR / "qwen3vl_yolo_ocr_silver_lora_final"
trainer.model.save_pretrained(final_dir)
processor.save_pretrained(final_dir)
(final_dir / "rukopys_prompt_config.json").write_text(
    json.dumps(
        {
            "mode": "ocr_only_yolo_crop",
            "crop_prompts": CROP_PROMPTS,
            "max_pixels_crop": MAX_PIXELS_CROP,
            "text_types": sorted(TEXT_TYPES),
            "jitter_variants_per_region": JITTER_VARIANTS_PER_REGION,
        },
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)
print("Saved OCR LoRA adapter to", final_dir)
